# CircuitGraft — Phase 4 Real Experiment Matrix on Google Colab

This notebook executes the **real Phase 4 experiment matrix** on `CohereLabs/tiny-aya-global` using a CUDA GPU (Tesla T4 / L4 / A100).

### Features & Safeguards
1. **Google Drive Persistence:** All artifacts, manifests, and checkpoints are stored directly in your Google Drive (`circuit_graft_phase4/`). If Colab disconnects, your progress is saved and stages resume automatically.
2. **Memory Headroom & Swapfile:** Allocates a 10 GB disk swapfile to prevent Linux OOM kills during 3.5B weight processing.
3. **Git-Backed Traceability:** Runs off your GitHub clone (`0dvsh0/circuit-graft`) so manifests record authentic Git SHAs.
4. **Calibrated Execution:** Includes a quick 15-minute calibration check before running the multi-stage matrix.

> **Prerequisites (same HF account that owns `HF_TOKEN`):** accept the licences for `CohereLabs/tiny-aya-global`, `CohereLabs/tiny-aya-base`, and `allenai/wildguard`, plus the gated evaluation datasets `walledai/HarmBench`, `walledai/AdvBench`, `sorry-bench/sorry-bench-202503`, `allenai/wildjailbreak`, and `walledai/XSTest`.

> **Human review:** `domain_benign` and `domain_harmful` stay candidates until every row in their `human_review.csv` receives a `review_label`. The notebook stops after data curation until Gate G1 passes.

> **GPU:** drift uses `configs/drift_t4.yaml` (micro-batch 2 x accum 16, effective batch 32 -> ~1.5-2.5 h on a T4); discovery uses the logged `batch_size: 1` / `max_seq_len: 96` fp16 shape. Grafting loads three 3.5B checkpoints, so an L4/A100 high-RAM runtime is preferable.

## 1. Mount Google Drive and Configure GPU & Swap Memory

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Persistent storage directory on Drive
DRIVE_RUN_DIR = '/content/drive/MyDrive/circuit_graft_phase4'
os.makedirs(f'{DRIVE_RUN_DIR}/artifacts', exist_ok=True)
os.makedirs(f'{DRIVE_RUN_DIR}/results', exist_ok=True)
print('Google Drive ready at:', DRIVE_RUN_DIR)

# Allocate 10GB swap to protect against Linux OOM-killer during 3.5B model weight manipulation
!fallocate -l 10G /swapfile 2>/dev/null && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile || true
!free -h
!nvidia-smi


## 2. Clone Git Repository and Symlink Drive Persistence

In [ ]:
import os
import shutil

%cd /content
if not os.path.exists('/content/circuit-graft'):
    !git clone https://github.com/0dvsh0/circuit-graft.git /content/circuit-graft

%cd /content/circuit-graft
!git pull

# Symlink artifacts and results to Google Drive for permanent persistence
!rm -rf /content/circuit-graft/artifacts /content/circuit-graft/results
!ln -s /content/drive/MyDrive/circuit_graft_phase4/artifacts /content/circuit-graft/artifacts
!ln -s /content/drive/MyDrive/circuit_graft_phase4/results /content/circuit-graft/results
print('Symlinked artifacts and results to Google Drive successfully!')


## 3. Install Required Toolkits & Package

In [ ]:
# Pin the dependency surface resolved in requirements-main.lock.
# Do not reinstall torch: Colab's preinstalled CUDA build matches the GPU.
!pip install --upgrade pip
!pip install safetune==0.1.12 circuitkit==0.2.0 auditkit==1.1.3 "curatorkit[generation,hf]==0.1.22"
!pip install "transformers==5.22.1" "bitsandbytes==0.50.2" "peft==0.21.2" "trl==1.14.2" "transformer-lens==3.8.0" "datasets==5.1.0" "accelerate==1.15.0" "huggingface-hub==1.33.0" sentence_transformers litellm python-dotenv typer rich

# PEFT >=0.20 rejects torchao <0.16.0, but Colab ships torchao 0.10.0.
# Uninstalling torchao is the documented project workaround (SPEC.md).
!pip uninstall -y torchao
!pip install -e . --no-deps

import subprocess, sys
check = subprocess.run(
    [sys.executable, "-c",
     "import importlib.util; assert importlib.util.find_spec('torchao') is None, 'torchao is still installed'; "
     "import peft, transformers, torch; "
     "print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
if check.returncode != 0:
    raise RuntimeError("Dependency check failed:\n" + check.stderr)

!python -m circuitgraft.cli doctor


## 4. Set Authentication Secrets (Hugging Face & Cohere)

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    cohere_key = userdata.get('COHERE_API_KEY')
except Exception:
    hf_token, cohere_key = None, None

if not hf_token:
    hf_token = getpass('Enter Hugging Face Token: ')
if not cohere_key:
    # Optional: the verified data snapshot is imported from the HF Hub and the
    # evaluation judge is the logged string-match deviation, so Colab does not
    # call the Cohere API for generation or judging.
    print('COHERE_API_KEY not set (optional; not needed for the snapshot-based run)')

os.environ['HF_TOKEN'] = hf_token.strip()
os.environ['COHERE_API_KEY'] = cohere_key.strip()
os.environ['HF_ORG'] = 'dubeydevansh'

# Check Tiny Aya licence access
!python -m circuitgraft.cli doctor --check-models

# Fail early if any gated dataset or the WildGuard judge is inaccessible.
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
required = [
    ('CohereLabs/tiny-aya-global', 'model'),
    ('CohereLabs/tiny-aya-base', 'model'),
    ('allenai/wildguard', 'model'),
    ('walledai/HarmBench', 'dataset'),
    ('walledai/AdvBench', 'dataset'),
    ('sorry-bench/sorry-bench-202503', 'dataset'),
    ('allenai/wildjailbreak', 'dataset'),
    ('walledai/XSTest', 'dataset'),
]
blocked = []
for repo_id, repo_type in required:
    try:
        api.list_repo_files(repo_id, repo_type=repo_type)
    except Exception as exc:
        blocked.append(f'{repo_id} ({type(exc).__name__})')
        print(f'BLOCKED  {repo_id}: {exc}')
    else:
        print(f'OK       {repo_id}')

if blocked:
    raise RuntimeError(
        'Accept the licence/terms for these repositories with the SAME account that owns '
        'HF_TOKEN, then re-run this cell:\n  - ' + '\n  - '.join(blocked)
    )


## 5. Quick Colab Calibration Pass (Smoke Test)

In [ ]:
# Quick ~15 min run on CUDA to ensure all toolkits and VRAM monkeypatches are operating normally
!python scripts/d1_smoke.py --device cuda --output-dir results/d1/smoke --skip-existing


## 6. Stage 1 (Data Curation)

The data stage writes the SFT set, contrastive pairs, and evaluation sets. Generated `domain_benign` / `domain_harmful` rows are candidates until the human-review CSVs are complete. Re-running this cell reuses the generated candidates so review labels can be applied without sampling a new set. Delete `artifacts/data/curation/domain_*` to force fresh generation.

In [ ]:
# Preferred path: import the curated + human-verified data snapshot from the
# HF Hub (run_id data-bc99df1d4f173b72, gate G1 passed) instead of re-running
# generation + human review on Colab. If the import fails, the data-stage cell
# below runs and the G1 gate enforces human verification as usual.
import json
import os
from pathlib import Path

from huggingface_hub import snapshot_download

DATA_REPO = os.environ.get('CG_DATA_REPO', 'dubeydevansh/circuitgraft-phase4-data')
summary_path = Path('artifacts/data/summary.json')

if not summary_path.exists():
    try:
        snapshot_download(
            repo_id=DATA_REPO,
            repo_type='dataset',
            local_dir='artifacts/data',
            token=os.environ['HF_TOKEN'],
        )
        print('Imported curated data snapshot from', DATA_REPO)
    except Exception as exc:
        print(f'WARN: snapshot import failed ({type(exc).__name__}: {exc})')
        print('Falling back to the data-stage cell; the G1 gate below will enforce human review.')

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('data run_id:', summary.get('run_id'))
    print('G1:', summary.get('gate_g1', {}).get('status'))
    assert Path('artifacts/data/medmcqa_sft/train.jsonl').exists(), 'SFT rows missing'
    assert Path('artifacts/data/pairs/safety.jsonl').exists(), 'pairs missing'
    print('data snapshot ready; the data-stage cell will skip automatically.')


In [ ]:
%%bash
set -euo pipefail
# Skipped when the verified data snapshot was imported in the previous cell;
# otherwise runs the full curation + generation + human-review flow.
if [ -f artifacts/data/summary.json ]; then
  echo 'SKIP data stage: verified snapshot already present (artifacts/data/summary.json)'
else
  python -m circuitgraft.cli data --config configs/data.yaml
fi


In [ ]:
# Gate G1: all configured sets must exist and domain-sensitive rows must be human-verified.
import json
from pathlib import Path

summary_path = Path('artifacts/data/summary.json')
if not summary_path.exists():
    raise FileNotFoundError('data stage did not write artifacts/data/summary.json')
summary = json.loads(summary_path.read_text(encoding='utf-8'))

print('Evaluation sets:')
for name, info in summary.get('eval', {}).items():
    print(
        f"  {name:22s} {str(info.get('status')):32s} "
        f"rows={info.get('rows')} human_verified={info.get('human_verified')}"
    )
for kind in ('benign', 'harmful'):
    info = summary.get('generation', {}).get(f'domain_{kind}', {})
    if info.get('review_path'):
        print(f"  review {kind}: {info['review_path']}")

gate = summary.get('gate_g1', {})
print('\nG1:', gate.get('status'))
for reason in gate.get('reasons', []):
    print('  -', reason)

if gate.get('status') != 'passed':
    raise RuntimeError(
        'Data gate G1 has not passed. Fix blocked datasets and complete the human-review '
        'CSVs (review_label = benign/safe or harmful/unsafe), then re-run the data cell.'
    )
print('\nG1 passed; proceeding to drift.')


## 7. Stage 2 (Drift SFT) & Stage 3 (Baseline Audit)

Each cell runs under `bash -e`, so drift/audit stop immediately on the first failure instead of continuing with a missing checkpoint.

In [ ]:
%%bash
set -euo pipefail
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

echo '--- GPU preflight ---'
nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv || true
python - <<'PY'
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device 0:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no CUDA - drift would run on CPU and be unusably slow')
PY

# configs/drift_t4.yaml = micro-batch 2 x accum 16 (effective batch 32 ->
# identical LR schedule and 314 optimizer steps). Micro-batch 4 OOM'd on the
# 14.56 GiB T4: the fp32 CE upcast over the 262144-token vocab at seq 1024
# needs a 4.29 GiB buffer on top of the fp16 logits; batch 2 halves that
# stack (~10.5 GiB peak). Falls back to the 8 GB RX 6600 profile when absent.
DRIFT_CONFIG=configs/drift.yaml
if [ -f configs/drift_t4.yaml ]; then DRIFT_CONFIG=configs/drift_t4.yaml; fi
echo "drift config: $DRIFT_CONFIG"
python -m circuitgraft.cli drift --config "$DRIFT_CONFIG"
echo '=== DRIFT COMPLETE (checkpoints + merged model written) ==='


In [ ]:
%%bash
set -euo pipefail
# S3 audit of aligned vs drifted. If this cell fails, the drift checkpoints
# are already saved (auto-resume) - only the audit needs re-running.
python -m circuitgraft.cli audit --config configs/eval.yaml --models aligned,drifted
echo '=== AUDIT COMPLETE ==='


## 8. Stage 4 (Dual-Circuit Discovery)

`configs/discover.yaml` carries the logged prereg deviation for the 8 GB card:
`dtype: float16`, `max_seq_len: 96`, `batch_size: 1` (pairs are short: safety
max 81 tokens; domain median 48, max 206). The runner also reties
`unembed.W_U` to `embed.W_E.T` when `logit_scale == 1.0`, which frees a
duplicate ~1 GiB untied LM head on load (see `_retie_unembed` in
`circuitkit_runner.py`). On a 15 GB T4 this shape runs with large headroom;
`ig_steps: 5` x 256 pairs is the main time cost. If it still OOMs, use an
L4/A100 runtime.


In [ ]:
%%bash
set -euo pipefail
python -m circuitgraft.cli discover --config configs/discover.yaml


## 9. Stage 5 (Baselines) & Stage 6 (Circuit Grafting)

Grafting loads the base, aligned, and drifted checkpoints together. On a 15 GB T4 watch system RAM and swap; a high-RAM runtime is strongly recommended.

Runs `cg_project`, `cg_uniform`, `cg_exclude`, `cg_distributed`, and `cg_random`. `cg_fisher` runs only if its Fisher artifact exists; otherwise it is skipped as the first allowed scope cut in SPEC.md and must be disclosed.

In [ ]:
%%bash
set -euo pipefail
python -m circuitgraft.cli baselines --config configs/baselines/resta.yaml
python -m circuitgraft.cli baselines --config configs/baselines/wiseft.yaml
python -m circuitgraft.cli baselines --config configs/baselines/ctheta.yaml
python -m circuitgraft.cli baselines --config configs/baselines/safelora.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_project.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_uniform.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_exclude.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_distributed.yaml
python -m circuitgraft.cli graft --config configs/graft/cg_random.yaml

# cg_fisher is the first allowed scope cut (SPEC.md). Run it only when
# the domain Fisher artifact exists; otherwise leave it out and disclose it.
if [[ -f artifacts/graft/fisher/domain_fisher.safetensors ]]; then
    python -m circuitgraft.cli graft --config configs/graft/cg_fisher.yaml
else
    echo 'SKIP cg_fisher: artifacts/graft/fisher/domain_fisher.safetensors is absent (declared scope cut)'
fi


## 10. Stage 7 (Comprehensive Evaluation) & Stage 8 (Attacks)

In [ ]:
%%bash
set -euo pipefail
python -m circuitgraft.cli eval --config configs/eval.yaml --all
python -m circuitgraft.cli attack --config configs/attacks.yaml --type abliterate
python -m circuitgraft.cli eval-attacks --config configs/eval.yaml


## 11. Stage 9 (Reporting & README) & Stage 10 (Publish to HF)

In [ ]:
%%bash
set -euo pipefail
python -m circuitgraft.cli report --output-dir results/s9 --readme README.md
python -m circuitgraft.cli publish
